# 01 — Data Collection & Document Parsing

**Series:** RAG for Financial & Legal Documents  
**Article:** «У меня есть документы. С чего начать?»

---

## What you will learn in this notebook

1. **Why naive approaches fail** — why you can't just dump documents into an LLM
2. **Document types and their parsing challenges** — PDF (text vs scanned), DOCX, tables
3. **Generate synthetic data** — realistic fake financial documents we'll use throughout the series
4. **From Scratch parsers** — ~80 lines that show exactly what happens under the hood
5. **LangChain loaders** — the same result in ~15 lines
6. **Metadata schema** — why you need it from day one

Each code section is built **from scratch first**, then repeated with **LangChain**.  
At the end of each section: a verdict on which approach to use in production.

---

## The "lost in the middle" problem

Before writing a single line of code, let's answer the obvious question:

> *Can't we just paste everything into ChatGPT and ask questions?*

Three reasons this fails at scale:

1. **Context window limits.** GPT-4o supports ~128k tokens. One financial report is ~30 pages = ~15k tokens. Ten clients × five documents each = 750k tokens. Doesn't fit.

2. **Cost.** Sending 750k tokens *per question* costs ~$0.75 per query. At 1000 questions/day that's $750/day on context alone.

3. **"Lost in the middle" effect.** Even when everything fits, LLMs recall information from the beginning and end of context much better than the middle. Your most important clause on page 7 may be effectively invisible.

**RAG solves this** by retrieving only the relevant fragments before generation — typically 3–10 chunks out of thousands.

---

## Architecture overview

```
OFFLINE PIPELINE (runs once when documents are ingested)
────────────────────────────────────────────────────────
Documents (PDF, DOCX)
    │
    ▼
[Parsing]      ← THIS NOTEBOOK
    │  Extract text, tables, metadata
    ▼
[Chunking]     ← Notebook 02
    │  Split into retrievable fragments
    ▼
[Embedding]    ← Notebook 03
    │  Convert text → vectors
    ▼
[Indexing]     ← Notebook 03
       Store in Qdrant vector DB

ONLINE PIPELINE (runs on every user query)
────────────────────────────────────────────────────────
User Question
    │
    ▼
[Retrieval]    ← Notebook 03 & 04
    │  Find the most relevant chunks
    ▼
[Generation]   ← Notebook 03
       LLM answers using only those chunks
```

Today we are building the very first box: **Parsing**.

## Part 0 — Setup

Make sure you have:
1. Created `.env` from `.env.example` and added your `OPENAI_API_KEY`
2. Run `make setup` (or `pip install -e ".[dev]"`)

The cell below adds the project root to `sys.path` so we can import from `src/`.

In [ ]:
import sys
from pathlib import Path

# Add project root to path so `from src.xxx import yyy` works
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import settings
print(f"Project root : {project_root}")
print(f"LLM model    : {settings.llm_model}")
print(f"Embedding    : {settings.embedding_model}")
print(f"Qdrant URL   : {settings.qdrant_url}")

## Part 1 — Generate Synthetic Data

### Why synthetic data?

We need realistic documents, but we can't use real client data (confidentiality).  
Synthetic data gives us two things real data can't:

- **Known ground truth** — we generated the revenue figures, so we know the correct answers. This lets us build a proper evaluation dataset in Notebook 05.
- **Controlled complexity** — we can design documents to test specific failure modes.

### What we generate

| Client | Documents |
|--------|----------|
| **Acme Corp** (B2B SaaS) | Annual reports 2022, 2023 · Service contract · Invoices |
| **Globex Inc** (Manufacturing) | Annual reports 2022, 2023 · NDA · MSA |
| **Initech LLC** (Fintech startup) | Financial projections · Term sheet |
| Internal | Risk Management Policy · KYC/AML Procedures |

### How it works

```
Prompt → GPT-4o-mini → JSON with sections & tables
                              │
                              ▼
                         reportlab
                              │
                              ▼
           Formatted PDF (header, footer, page numbers, styled tables)
```

Generating the PDFs with `reportlab` is itself interesting content: it shows how to produce documents that *look* like real corporate PDFs, with proper table styling, headers, and footers.

**⚠️ This cell calls the OpenAI API and will cost a small amount (~$0.05–0.10 total).  
Run it once; use `--skip-llm` on subsequent runs to reuse cached responses.**

In [ ]:
import subprocess
result = subprocess.run(
    ["python", str(project_root / "scripts" / "generate_synthetic_data.py")],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# List all generated documents
raw_dir = project_root / "data" / "raw"
pdf_files = sorted(raw_dir.rglob("*.pdf"))
print(f"Generated {len(pdf_files)} PDF documents:\n")
for f in pdf_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.relative_to(project_root)}  ({size_kb:.1f} KB)")

## Part 2 — Why PDF Parsing Is Harder Than It Looks

PDF stands for *Portable Document Format* — designed for faithful visual rendering, **not** for text extraction.  
There is no concept of "paragraphs" or "tables" at the format level. There are only drawing instructions: *draw character X at position (123, 456) with font Y and size Z*.

This means that extracting readable text requires **reconstructing** the logical structure from raw drawing coordinates. That's what libraries like PyMuPDF do.

### Common failure modes

| Problem | What happens |
|--------|-------------|
| Multi-column layout | Columns get merged left-to-right producing garbled text |
| Headers / footers | Page numbers and document titles pollute every chunk |
| Tables | Cells collapse into a single string with no delimiters |
| Scanned PDFs | No text layer at all — just a bitmap image (needs OCR) |
| Hyphenation | "reve-\nnue" instead of "revenue" |

### Our strategy

- **PyMuPDF (fitz)** — fast, reliable main-text extraction. Preserves block reading order.
- **pdfplumber** — slower but understands table geometry. Extracts cells as structured rows.
- The two libraries complement each other. We use both.

## Part 3 — From Scratch: PDF Parser

Let's build the parser step by step, so you can see exactly what each library does.

First, a direct look at what PyMuPDF gives us raw:

In [ ]:
import fitz  # PyMuPDF

pdf_path = project_root / "data/raw/clients/acme_corp/financial_report_2023.pdf"

doc = fitz.open(str(pdf_path))
print(f"Pages: {len(doc)}")
print(f"\n--- Page 1 raw text (first 800 chars) ---\n")

page = doc[0]
raw_text = page.get_text("text")
print(raw_text[:800])

doc.close()

In [ ]:
import pdfplumber

# Now see what pdfplumber finds for tables
with pdfplumber.open(str(pdf_path)) as pdf:
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            print(f"\n--- Page {i+1}: found {len(tables)} table(s) ---")
            for t_idx, table in enumerate(tables):
                print(f"  Table {t_idx+1}: {len(table)} rows × {len(table[0])} cols")
                print(f"  Header: {table[0]}")
                print(f"  Row 1:  {table[1] if len(table) > 1 else 'N/A'}")

In [ ]:
# Now use our complete parser from src/
from src.from_scratch.ingestion.parsers.pdf_parser import parse_pdf

doc = parse_pdf(
    pdf_path,
    metadata={
        "client_id": "acme_corp",
        "doc_type": "financial_report",
        "year": 2023,
        "confidentiality": "confidential",
    },
)

print(f"Pages        : {doc.num_pages}")
print(f"Tables found : {len(doc.all_tables)}")
print(f"Metadata     : {doc.metadata}")
print(f"\n--- full_text preview (first 600 chars) ---\n")
print(doc.full_text[:600])

In [ ]:
from src.from_scratch.ingestion.parsers.table_parser import make_table_chunk

if doc.all_tables:
    chunk = make_table_chunk(
        raw_table=doc.all_tables[0],
        metadata=doc.metadata,
        title="Revenue Breakdown",
    )
    print("--- Table as embeddable markdown ---\n")
    print(chunk.text_representation)
    print(f"\nMetadata: {chunk.metadata}")
else:
    print("No tables found on this document.")

## Part 4 — From Scratch: DOCX Parser

DOCX is much friendlier. The format is an open XML spec, and `python-docx` gives us clean programmatic access to:

- **Paragraphs** — with their style name (Normal, Heading 1, Heading 2, ...)
- **Tables** — as a grid of Cell objects
- **Runs** — individual text segments with formatting

Our parser converts heading styles to markdown markers (`# `, `## `) so that chunkers can later use headings as natural split boundaries.

In [ ]:
import docx

# Note: our synthetic data generates PDFs, not DOCX.
# If you have a real DOCX, point this to it.
# For demo purposes we'll create a tiny in-memory DOCX.
from io import BytesIO
import docx as python_docx

buf = BytesIO()
demo_doc = python_docx.Document()
demo_doc.add_heading("Risk Management Policy", level=1)
demo_doc.add_heading("1. Purpose and Scope", level=2)
demo_doc.add_paragraph(
    "This policy establishes the framework for identifying, assessing, "
    "and managing risks across all business operations."
)
demo_doc.add_heading("2. Risk Categories", level=2)
demo_doc.add_paragraph("We recognise four primary risk categories: market, credit, operational, and regulatory.")

# Add a small table
tbl = demo_doc.add_table(rows=3, cols=3)
headers = ["Risk Level", "Probability", "Action"]
rows_data = [("High", "Frequent", "Escalate immediately"), ("Low", "Rare", "Monitor quarterly")]
for i, h in enumerate(headers):
    tbl.rows[0].cells[i].text = h
for r_idx, row in enumerate(rows_data):
    for c_idx, val in enumerate(row):
        tbl.rows[r_idx + 1].cells[c_idx].text = val

demo_doc.save(buf)
buf.seek(0)

# Save to disk so our parser can open it
demo_docx_path = project_root / "data" / "raw" / "policies" / "demo_risk_policy.docx"
demo_docx_path.parent.mkdir(parents=True, exist_ok=True)
with open(demo_docx_path, "wb") as f:
    f.write(buf.read())
print(f"Demo DOCX written to: {demo_docx_path}")

In [ ]:
from src.from_scratch.ingestion.parsers.docx_parser import parse_docx

docx_doc = parse_docx(
    demo_docx_path,
    metadata={
        "doc_type": "policy",
        "confidentiality": "internal",
    },
)

print("--- Parsed text (headings as markdown) ---\n")
print(docx_doc.text)
print(f"\n--- Tables: {len(docx_doc.tables)} ---")
for i, tbl in enumerate(docx_doc.tables):
    print(f"  Table {i+1}: {len(tbl)} rows")
    for row in tbl:
        print(f"    {row}")

## Part 5 — Metadata Schema

### Why metadata matters

Imagine our system stores documents from Acme Corp and Globex Inc.  
A user asks: *"What was the revenue in Q3 2023?"*

Without metadata filtering, the system might return chunks from **both** clients — mixing numbers that don't belong together.

Metadata enables **pre-filtering**: search *only within* Acme Corp's financial reports for 2023.  
This is one of the most impactful accuracy improvements you can make.

### Schema design

Every document chunk in Qdrant will carry these fields:

```python
{
    # --- Identity ---
    "client_id":       "acme_corp",          # or None for internal docs
    "doc_type":        "financial_report",   # financial_report | contract | policy | nda | ...
    "year":            2023,
    "quarter":         "Q3",                 # if applicable
    
    # --- Access control ---
    "confidentiality": "confidential",       # public | internal | confidential
    
    # --- Location (for citations) ---
    "source_file":     "acme_corp/financial_report_2023.pdf",
    "page":            4,
    "section":         "Revenue Breakdown",  # if detectable
    
    # --- Content type ---
    "content_type":    "text",               # text | table
    "chunk_index":     12,                   # position within the document
}
```

**Rule:** Design your metadata schema before you write a single chunking or embedding line.  
Changing it later means re-indexing everything from scratch.

In [ ]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class DocumentMetadata:
    """
    Typed metadata schema for all documents in the system.
    Using a dataclass enforces the schema and catches typos early.
    """
    source_file: str
    doc_type: Literal[
        "financial_report", "contract", "nda", "policy",
        "financial_projections", "kyc", "invoice", "other"
    ]
    confidentiality: Literal["public", "internal", "confidential"]
    client_id: str | None = None
    year: int | None = None
    quarter: str | None = None
    page: int | None = None
    section: str | None = None
    content_type: Literal["text", "table"] = "text"
    chunk_index: int | None = None

    def to_dict(self) -> dict:
        """Serialize for storage in Qdrant payload."""
        return {k: v for k, v in self.__dict__.items() if v is not None}


# Example: Acme Corp's 2023 financial report
meta = DocumentMetadata(
    source_file="clients/acme_corp/financial_report_2023.pdf",
    doc_type="financial_report",
    confidentiality="confidential",
    client_id="acme_corp",
    year=2023,
    page=4,
    section="Revenue Breakdown",
    content_type="table",
    chunk_index=12,
)

print("Metadata dict (Qdrant payload):")
import json
print(json.dumps(meta.to_dict(), indent=2))

## Part 6 — LangChain: The Same Result in ~15 Lines

Now let's do the same thing with LangChain's document loaders.

The key abstraction is the `Document` object:

```python
Document(
    page_content="...",    # the text
    metadata={...}          # any key-value pairs
)
```

This is the standard interface for the entire LangChain pipeline. The same object flows from loader → splitter → embedder → vector store. You never need to convert or adapt.

Compare to our from-scratch `ParsedDocument`: it has the same information, but a custom shape. To feed it into LangChain later, we'd need an adapter. LangChain's `Document` gives us that compatibility for free.

In [ ]:
from src.langchain_impl.ingestion.loaders import load_pdf

lc_docs = load_pdf(
    pdf_path,
    metadata={
        "client_id": "acme_corp",
        "doc_type": "financial_report",
        "year": 2023,
        "confidentiality": "confidential",
    },
)

print(f"Loaded {len(lc_docs)} LangChain Documents (one per page)")
print(f"\nDocument type: {type(lc_docs[0]).__name__}")
print(f"Metadata     : {lc_docs[0].metadata}")
print(f"\nContent preview (page 1):")
print(lc_docs[0].page_content[:500])

In [ ]:
from src.langchain_impl.ingestion.loaders import load_directory

# Load all Acme Corp PDFs in one call
acme_docs = load_directory(
    project_root / "data/raw/clients/acme_corp",
    glob="**/*.pdf",
    metadata_fn=lambda path: {"client_id": "acme_corp"},
)

print(f"Loaded {len(acme_docs)} Documents from Acme Corp")
# Each document already has client_id in metadata from our metadata_fn
for d in acme_docs[:3]:
    print(f"  {d.metadata.get('source', '')!r}  →  page {d.metadata.get('page', '?')}")

## Part 7 — Comparison & Verdict

| Aspect | From Scratch | LangChain |
|--------|-------------|----------|
| Lines of code | ~80 per parser | ~15 total |
| Table extraction | ✅ pdfplumber gives structured rows | ❌ Not available |
| Per-page granularity | ✅ Custom `ParsedPage` | ✅ PyPDFLoader returns one Doc/page |
| Metadata | Manual | Auto (source, page) + custom |
| Downstream compatibility | Custom, needs adapter | ✅ Works directly with splitters, vector stores |
| Complex layouts | Manual tuning possible | Depends on loader |

### When to use each

**Use LangChain loaders when:**
- You need a quick prototype or the document structure is standard
- You want the result to flow directly into LangChain's splitters and vector stores
- The documents are text-heavy without complex tables

**Use the custom from-scratch parser when:**
- You need structured table data (financial reports, data sheets)
- You need custom metadata extraction (e.g., parse the contract number from the header)
- You need fine-grained control over what gets included (strip footers, handle multi-column)

**In this project:** we use the custom parser for financial reports (tables matter), LangChain for policies and contracts (text-heavy, standard structure).

In [ ]:
# Final summary: what we parsed
print("=" * 55)
print("  PARSING SUMMARY")
print("=" * 55)

all_pdfs = sorted((project_root / "data" / "raw").rglob("*.pdf"))
total_pages = 0
total_tables = 0

for pdf in all_pdfs:
    try:
        parsed = parse_pdf(pdf)
        total_pages += parsed.num_pages
        total_tables += len(parsed.all_tables)
        print(f"  {pdf.relative_to(project_root)}")
        print(f"    Pages: {parsed.num_pages}  |  Tables: {len(parsed.all_tables)}")
    except Exception as e:
        print(f"  [ERROR] {pdf.name}: {e}")

print("-" * 55)
print(f"  Total PDFs  : {len(all_pdfs)}")
print(f"  Total pages : {total_pages}")
print(f"  Total tables: {total_tables}")
print("=" * 55)
print()
print("Next: Notebook 02 — Chunking Strategies")

## What's Next

We now have parsed documents: text extracted page by page, tables in structured form, metadata attached.

The next question is: **how do we split this text into chunks?**

A chunk too small has no context. A chunk too large adds noise. The wrong boundary can split a sentence right in the middle of a key financial figure.

In **Notebook 02** we implement three strategies:
- Fixed-size chunking with overlap (simple, fast)
- Semantic chunking (split by meaning, not by character count)
- Document-aware chunking (respect headings and section structure)

...and show concrete examples where each wins and where it fails.

---
*Article: «Как разбить документы на куски?»*